# Loading script for Green stay hotels in DWHPFX schema

In [24]:
import requests
import json
import pandas as pd
import time
import configparser
import pyexasol

In [35]:
def get_token(urlLogin, env):
    """ This function authenticates and returns token and refreshToken """
    auth = {"client_id": "00000000-0000-0000-0000-000000000000", "client_secret": "00000000-0000-0000-0000-000000000000"}
    response = requests.post(urlLogin, json=auth)
    if response.status_code == 200:
        data = response.text
        parsed = json.loads(data)
        token = parsed['token']
        refreshToken = parsed['refreshToken']
    return token, refreshToken

def GreenStayExtract(urlAuth, env='det'):
    """ This function fetches the data from the api and stores in a df """

    token, refreshToken = get_token(urlAuth, env)
    startTime = time.time()
    urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page=1&size=500"
    getResponse = requests.get(urlGet)
    rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                      'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                      'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                      'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                      'status': 'GREEN_INSPECTION_STATUS'
                      }

    if getResponse.status_code == 200:
        data = getResponse.text
        parsed = json.loads(data)
        print(f"Status code: {getResponse.status_code}")
    else:
        print("Error in fetching the pages")
        
    try:
        temp = []
        for x in range(int(parsed['total_pages'])):
            urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page={x+1}&size=500"
            getResponse = requests.get(urlIter)

            if getResponse.status_code == 200:
                data = getResponse.text
                parsed = json.loads(data)
                temp.append(parsed.get('results'))
            else:
                print("Request failed page: {} ".format(x))
        flatten = [item for sublist in temp for item in sublist]
        dfRaw = pd.DataFrame(flatten) 
        dfRaw['created_date'] = dfRaw['created_date'].astype(str).str[:-6]
        dfRaw['updated_date'] = dfRaw['updated_date'].astype(str).str[:-6]
        dfRaw['created_date'] = pd.to_datetime(dfRaw['created_date'], utc=False)
        dfRaw['updated_date'] = pd.to_datetime(dfRaw['updated_date'], utc=False)
        dfRaw = dfRaw[dfRaw.hkey.notnull()]
        dfRaw = dfRaw[['hkey', 'created_date', 'updated_date', 'report_year', 'kilogramCarbonPOC', 'literWaterPOC',
                       'kilogramWastePOC', 'carbonClass', 'waterClass', 'wasteClass', 'greenClass', 'type', 'status']]
        dfRaw.rename(columns=rename_columns, inplace=True)
        df = dfRaw.sort_values('UPDATED_DATE').groupby('HOTEL_ID').tail(1)
        df['CREATED_DATE'] = pd.to_datetime(dfRaw['CREATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        df['UPDATED_DATE'] = pd.to_datetime(dfRaw['UPDATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        df['GREEN_INSPECTION_TYPE'] = df['GREEN_INSPECTION_TYPE'].apply(lambda x: x.upper())
        df['LDTS'] = time.strftime('%Y-%m-%d')
        print(f'Number of records having HOTEL_IDs {dfRaw.HOTEL_ID.nunique()}.')
        print(f'Number of records with duplicate HOTEL_IDs: {dfRaw.duplicated(subset="HOTEL_ID", keep="first").sum()}.')
        endTime = time.time() - startTime
        print(f'Time in minutes: {round(endTime / 60, 2)}')
    except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function GreenScore - ')
        print(f'Time in minutes: {round(endTime / 60, 2)}')
        raise e
    return df


def GreenStayLoad(df): 
    """ This function is to load the data into Exasol DB -> DWHPFX schema """
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.GREEN_STAY_HOTELS")
        connect.import_from_pandas(df, table = ('DWHPFX','GREEN_STAY_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except Exception as e:
        print('Failed in function GreenStayLoad - ')
        raise e

### Function call

In [36]:
df=GreenStayExtract('https://api.hotel-audit.hrs.com/auth/login')
# GreenStayLoad(df)
df.head()

### Test results in excel

In [106]:
from datetime import date
df.to_excel('C:\\Users\\USER\\Documents\\misc\\'+str(date.today())+'_test.xlsx', 
              sheet_name='Sheet1', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )

In [14]:
import pandas as pd

data = [(-1, -1, 0), (-1, 1, 1), (1, -1, 1), (1, 1, 0)]

dftest = pd.DataFrame(data, columns=['feature1', 'feature2', 'label'])

In [15]:
dftest.corr()

In [11]:
lastname: str = 100

In [12]:
lastname

In [3]:
int = 'Lakshmi'

In [4]:
int

In [5]:
lastname :int

In [6]:
lastname

In [105]:
urlAuth='https://api.hotel-audit.hrs.com/auth/login'
token, refreshToken = get_token(urlAuth, env='det')
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report?token={token}&page=1&size=1"
getResponse = requests.get(urlGet)
rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                  'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                  'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                  'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                  'status': 'GREEN_INSPECTION_STATUS'
                  }

if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
    print(parsed['total_pages'])
    print(f"Status code: {getResponse.status_code}")
else:
    print("Error in fetching the pages")
# try:
#     temp=[]
#     for x in range(int(parsed['total_pages'])):
#         urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page={x+1}&size=500"
#         getResponse = requests.get(urlIter)

#         if getResponse.status_code == 200:
#             data = getResponse.text
#             parsed = json.loads(data)
#             temp.append(parsed.get('results').values())
#         else:
#             print("Request failed page: {} ".format(x))
#         flatten = [item for sublist in temp for item in sublist]
#         dfRaw = pd.DataFrame(flatten)
#         dfRaw['created_date'] = dfRaw['created_date'].astype(str).str[:-6]
#         dfRaw['updated_date'] = dfRaw['updated_date'].astype(str).str[:-6]
#         dfRaw['created_date'] = pd.to_datetime(dfRaw['created_date'], utc=False)
#         dfRaw['updated_date'] = pd.to_datetime(dfRaw['updated_date'], utc=False)
#         dfRaw = dfRaw[dfRaw.hkey.notnull()]
#         dfRaw = dfRaw[['hkey', 'created_date', 'updated_date', 'report_year', 'kilogramCarbonPOC', 'literWaterPOC',
#                        'kilogramWastePOC', 'carbonClass', 'waterClass', 'wasteClass', 'greenClass', 'type', 'status']]
#         dfRaw.rename(columns=rename_columns, inplace=True)
#         df = dfRaw.sort_values('UPDATED_DATE').groupby('HOTEL_ID').tail(1)
#         df['CREATED_DATE'] = pd.to_datetime(dfRaw['CREATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
#         df['UPDATED_DATE'] = pd.to_datetime(dfRaw['UPDATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
#         df['GREEN_INSPECTION_TYPE'] = df['GREEN_INSPECTION_TYPE'].apply(lambda x: x.upper())
#         df['LDTS'] = time.strftime('%Y-%m-%d')
#         print(f'Number of records having HOTEL_IDs {dfRaw.HOTEL_ID.nunique()}.')
#         print(f'Number of records with duplicate HOTEL_IDs: {dfRaw.duplicated(subset="HOTEL_ID", keep="first").sum()}.')
#         endTime = time.time() - startTime
#         print(f'Time in minutes: {round(endTime / 60, 2)}')
# except Exception as e:
#         endTime = time.time() - startTime
#         print('Failed in function GreenScore - ')
#         print(f'Time in minutes: {round(endTime / 60, 2)}')
#         raise e

In [106]:
parsed

In [55]:
urlAuth='https://api.hotel-audit.hrs.com/auth/login'
token, refreshToken = get_token(urlAuth, env='det')
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page=1&size=500"
getResponse = requests.get(urlGet)
rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                  'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                  'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                  'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                  'status': 'GREEN_INSPECTION_STATUS'
                  }

if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
#     print(parsed)
    print(f"Status code: {getResponse.status_code}")
else:
    print("Error in fetching the pages")
try:
    dfRaw = pd.DataFrame()
    temp1=[]
    for x in range(2):
        urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page={x+1}&size=500"
        getResponse = requests.get(urlIter)

        if getResponse.status_code == 200:
            data = getResponse.text
            parsed = json.loads(data)
            temp1.append(parsed.get('results'))
        else:
            print("Request failed page: {} ".format(x))
except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function GreenScore - ')
        print(f'Time in minutes: {round(endTime / 60, 2)}')
        raise e
temp1

In [67]:
temp1[1][0]